Things to add/areas for improvement
- simulation time step -> 1hr?, 30 mins?
      - smart grid smart city data uses 30 min intervals for household demand/usage, 48 timesteps per simulation scenario to simulate a full day
      - ensure generation profiles are using half hourly TMY data in SAM to match
-  realistic load/generation profiles for houses
      - solar curves
      - battery charge/discharge profliles
      - EV charge/discharge profiles -> dynamic and handled by the simulation rather than predefined profiles
      - these might be found from realworld sources/data tables, find smart meter data if possible -> found smart grid smart city meter data need to clean and select house profiles
      - should consider days where temperature and weather changes increase demand or decrease generation
            - check smart grid smart city data for location and date information could cross check with TMY data for that year/location
            - generation profiles can be handled in SAM
- realistic load profiles for industry/hospital
      - try and find realworld data to ensure daily curves are accurate
      - need more research to ensure realism
- Improve Windturbine simulation
      - Different weather profiles that show no wind/lockout, moderate and ideal wind (can use to show DFIG behaviour)
- can this each simulation run based on changning input settings?
      -confident the code so far can be adapted later to take inputs from external sources i.e buttons or removal of EV models
- data outputs that can be represented graphically and physically with leds etc
     - night/day time shift and change -> can this be visibly shown on display? e.g street lamps etc
     - 
- increasing complexity of power flow -> add pandapower validation layer
     - add things like power factor, transmission lines, transformers/substations and how these effect the simulation
     - how do inverters and their powerfactors effect the simulation
     - this will likely be handled by pandapower
- ADD VPP controller
     - start with controlling basic simulation layer
     - then add inputs and response from pandapower validation layer
     - Stage 1:
        Predefined event hours

     - Stage 2:
        Dataset-driven events
        hospital_load_kw exceeds threshold

     - Stage 3:
        User-input-driven events
        button/switch changes external_event

     - Stage 4:
        Network-driven events
        pandapower detects voltage/loading issue

     - Stage 5:
        Market-driven events
        price signal changes mode
       
- Financial moddelling
     - import/export tarif calculations
     - research indicates that VPPs allow for hoigher export tarifs than traditional residential PV exports -> integrate both models and show difference on data screen?
     - 

In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import os


#create results folder if it doesn't exist
os.makedirs("results", exist_ok=True)


print("Environment working")


Environment working


Define simulation Objects


In [20]:
class House:
    def __init__(self, name, load_profile, generation_profile, battery=None):
        self.name = name
        self.load_profile = load_profile
        self.generation_profile = generation_profile
        self.battery = battery

    def step(self, t, timestep_hours, signals):
        load_kw = self.load_profile[t]
        generation_kw = self.generation_profile[t]
        # check if there is a surplus of generation
        net_power_kw_bb = load_kw - generation_kw
        # initialise battery power as 0 incase load meets demand
        battery_power_kw = 0.0
        # initialise battery_soc_kwh and percent incase no battery detected on house these values are local only and the true values exist within the objects themselves
        battery_soc_kwh = 0.0
        battery_soc_percent = 0.0
        
        
        # check to see if there is a battery apart of this household
        if self.battery is not None:
            # check what mode the VPP controller is instructing and execute actions
            mode = signals["mode"]
            is_peak_period = signals["is_peak_period"]
            is_event_period = signals["is_event_period"]
            is_cheap_energy_period = signals["is_cheap_energy_period"]
            
            if mode == "self_consumption":
            
                if net_power_kw_bb < 0:
                # run charge function for power surplus
                    battery_power_kw = self.battery.charge(-net_power_kw_bb, timestep_hours)

                # else discharge battery to meet demand
                elif net_power_kw_bb > 0:
                    battery_power_kw = self.battery.discharge(net_power_kw_bb, timestep_hours)
            
               
            elif mode == "battery_hold":
                battery_power_kw = 0.0
                
            elif mode == "peak_shaving":
                # check to see if there is a surplus
                if net_power_kw_bb < 0:
                    # charge battery if there is a PV surplus
                    battery_power_kw = self.battery.charge(-net_power_kw_bb, timestep_hours)
                #Else check to see if it is within the peak period to shave demand
                elif net_power_kw_bb > 0 and is_peak_period:
                    # discharge battery if demand is greater than generation and within peak period time
                    battery_power_kw = self.battery.discharge(net_power_kw_bb, timestep_hours)
                else:
                    # battery is idle
                    battery_power_kw = 0.0
            elif mode == "export_support":
                battery_power_kw = self.battery.discharge(self.battery.max_discharge_kw, timestep_hours)
                # NOT USED CURRENTLY, ACTIVATE WHEN ECONOMIC MODEELING IS IMPLEMENTED
    
            elif mode == "local_load_support":
                if is_event_period:
                    battery_power_kw = self.battery.discharge(self.battery.max_discharge_kw, timestep_hours)
                    # need to improve later to pass target for specific house contribution i.e contribution = request/number of batteries availiable
                        # signals = {
                        #     "mode": "local_load_support",
                        #     "house_dispatch_targets": {
                        #         "House 1": 5.0,
                        #         "House 2": 2.0,
                        #         "House 4": 3.0,
                        #     }
                        # }
                     # this will not work here but is what the signals function will need to pass later on
                     # target_kw = signals["house_dispatch_targets"].get(self.name, 0.0)
                else:
                    battery_power_kw = 0.0
                    # does this need to change to support charging outside of event hours or will/is this handled by the VPP logic changing mode?
                  
            elif mode == "arbitrage_charge":
                if is_cheap_energy_period:
                    battery_power_kw = self.battery.charge(self.battery.max_charge_kw, timestep_hours)
                
                
            else: 
                battery_power_kw = 0.0
            # update battery SOC values
            battery_soc_kwh = self.battery.get_soc_kwh()
            battery_soc_percent = self.battery.get_soc_percent()
            
        battery_discharge_kw = max(battery_power_kw, 0.0)
        battery_charge_kw = -min(battery_power_kw, 0.0)
        final_net_power_kw = load_kw - generation_kw - battery_power_kw   

        return {
            "name": self.name,
            "load_kw": load_kw,
            "generation_kw": generation_kw,
            "net_power_kw_before_battery": net_power_kw_bb,
            "battery_power_kw": battery_power_kw,
            "battery_soc_kwh": battery_soc_kwh,
            "battery_soc_percent": battery_soc_percent,
            "net_power_kw" : final_net_power_kw,
            "battery_charge_kw": battery_charge_kw,
            "battery_discharge_kw": battery_discharge_kw,
        }
#how do I improve load and generation simulation? need realistic load and genereation profiles for each house depending on their availiable resources. could also reseach files to obtain accurrate hourly load profiles (clean data in Excel)

# Create the Battery Class/Object, This may later be moved inside of the house class once tested for behaviour

class Battery:
    def __init__(self, capacity_kwh, initial_soc_kwh, max_charge_kw, max_discharge_kw, efficiency=0.95, min_soc_kwh=0.0):
        self.capacity_kwh = capacity_kwh
        self.soc_kwh = initial_soc_kwh
        self.max_charge_kw = max_charge_kw
        self.max_discharge_kw = max_discharge_kw
        self.efficiency = efficiency
        self.min_soc_kwh = min_soc_kwh
        
    def charge(self, requested_power_kw, timestep_hours):
        # check requested charge power is positive
        if requested_power_kw <= 0:
            return 0.0
        # limit requested power
        charge_power_kw = min(requested_power_kw, self.max_charge_kw)
        # convert to stored energy
        requested_energy_kwh = charge_power_kw *timestep_hours*self.efficiency
        # check availiable battery space
        available_space_kwh = self.capacity_kwh - self.soc_kwh
        # limit actual stored energy to within max capacity
        actual_energy_kwh = min(requested_energy_kwh, available_space_kwh)
        # update the SOC
        self.soc_kwh += actual_energy_kwh
        # calculate the charging power
        actual_charge_power_kw = actual_energy_kwh / (timestep_hours*self.efficiency)
        #return value of power stored by the battery to the simulation
        battery_power_kw = actual_charge_power_kw*-1
        return battery_power_kw
        
    def discharge(self, requested_power_kw, timestep_hours):
         # check requested discharge power is positive
        if requested_power_kw <= 0:
            return 0.0
        # limit requested discharge power
        discharge_power_kw = min(requested_power_kw, self.max_discharge_kw)
        # convert requested power to battery energy removed
        requested_energy_kwh = discharge_power_kw * timestep_hours / self.efficiency
        # check current SOC to determine if battery has available capacity
        available_energy_kwh = max(0.0, self.soc_kwh - self.min_soc_kwh)
        # limit discharged power to what is actually available
        actual_energy_kwh = min(available_energy_kwh, requested_energy_kwh) 
        # update the SOC
        self.soc_kwh -= actual_energy_kwh
        # calculate the discharge power
        actual_discharge_power_kw = (actual_energy_kwh*self.efficiency)/timestep_hours
        # return value of power discharged from the battery
        battery_power_kw = actual_discharge_power_kw
        return battery_power_kw
        
    def get_soc_percent(self):
        return (self.soc_kwh/self.capacity_kwh)*100
    def get_soc_kwh(self):
        return self.soc_kwh

# Define the VPP controller class
class VPPController:
    def __init__(self, 
                 default_mode="self_consumption", 
                 peak_hours=None, 
                 event_hours=None, 
                 cheap_energy_hours=None
                ):
        
        self.default_mode = default_mode
        self.peak_hours = peak_hours or []
        self.event_hours = event_hours or []
        self.cheap_energy_hours = cheap_energy_hours or []
        

    def get_signals(self, t):
        if t in self.event_hours:
            active_mode = "local_load_support"
        elif t in self.cheap_energy_hours:
            active_mode = "arbitrage_charge"
        elif t in self.peak_hours:
            active_mode = "peak_shaving"
        else:
            active_mode = self.default_mode

        return{
            "mode":active_mode,
            "is_peak_period":t in self.peak_hours,
            "is_event_period":t in self.event_hours,
            "is_cheap_energy_period":t in self.cheap_energy_hours
        }
        


#define a local load that the VPP can supply this is currently a basic place holder that will be updated later to match realistic demand values and have its own demand profile (currently static) 

class LocalLoad:
    def __init__(self, name, demand_kw):
        self.name = name
        self.demand_kw = demand_kw

    def step(self, t):
        return {
            "name": self.name,
            "load_kw": self.demand_kw
        }


class WindTurbine:
    def __init__(self, name, rated_kw):
        self.name = name
        self.rated_kw = rated_kw

    def step(self, t):
        generation_kw = 0.0  # placeholder for now
        return {
            "name": self.name,
            "generation_kw": generation_kw
        }

#wind turbine could have three different generation settings based on the wind speed (no wind/lock out (too much wind), moderate wind, ideal wind)
#middle setting could have variance to show slight hour to hour changes perhaps?
#need to confirm how this works in realworld -> further research required



Simulation Loop

In [28]:
# PLACEHOLDER LOAD/GENERATION PROFILES UNTIL REAL VALUES ARE CLEANED AND SELECTED

# these load profiles will be replaced with real world data from the smart grid smart city dataset

house_1_load = [0.7, 0.6, 0.6, 0.5, 0.6, 0.8, 1.4, 1.8, 1.2, 1.0, 0.9, 0.8,
                0.9, 1.0, 1.1, 1.3, 1.8, 2.6, 2.9, 2.4, 1.8, 1.3, 1.0, 0.8]

house_2_load = [0.9, 0.8, 0.7, 0.7, 0.8, 1.0, 1.8, 2.2, 1.5, 1.2, 1.1, 1.0,
                1.0, 1.1, 1.3, 1.6, 2.2, 3.1, 3.4, 2.8, 2.1, 1.6, 1.2, 1.0]

house_3_load = [0.5, 0.5, 0.4, 0.4, 0.5, 0.6, 1.0, 1.3, 0.9, 0.8, 0.7, 0.7,
                0.8, 0.8, 0.9, 1.0, 1.4, 2.0, 2.2, 1.8, 1.3, 1.0, 0.8, 0.6]

house_4_load = [1.0, 0.9, 0.8, 0.8, 0.9, 1.2, 2.0, 2.5, 1.8, 1.5, 1.3, 1.2,
                1.3, 1.4, 1.6, 1.9, 2.6, 3.7, 4.0, 3.3, 2.5, 1.9, 1.4, 1.1]

house_5_load = [0.8, 0.7, 0.7, 0.6, 0.7, 0.9, 1.6, 2.0, 1.4, 1.1, 1.0, 0.9,
                1.0, 1.1, 1.2, 1.5, 2.0, 2.9, 3.1, 2.6, 2.0, 1.5, 1.1, 0.9]
# these generation profiles will be replaced with PV generation profiles generated using an external tool such as SAM to represent different size systems and weather conditions

pv_sunny_large = [0, 0, 0, 0, 0, 0.2, 0.8, 1.8, 3.0, 4.0, 4.8, 5.2,
             5.0, 4.5, 3.6, 2.4, 1.2, 0.3, 0, 0, 0, 0, 0, 0]

pv_sunny_med = [x*0.5 for x in pv_sunny_large]

pv_cloudy = [0, 0, 0, 0, 0, 0.1, 0.4, 0.9, 1.5, 2.0, 2.3, 2.5,
             2.2, 2.0, 1.6, 1.0, 0.5, 0.1, 0, 0, 0, 0, 0, 0]

pv_none = [0]*24

#these values will change later as the smart grid smart city dataset is in half hourly readings but overall goal is for each profile to simulate a full day to show arbitrage effect of peak hour demand

profile_duration_hours = 24 #this will probably remain unchanged 

# following code ensures calculations later in simulations remain correct if length of datasets change but each profile should alwuts represent 1 day
num_steps = len(house_1_load)
timestep_hours = profile_duration_hours / num_steps
simulation_hours = num_steps * timestep_hours

#need to add Wind Turbine and industrial load profiles here potentially? UPDATE WIND GEN AND LOCAL LOAD CLASS WHEN THIS IS DONE

# Verify all profiles are the same length
profiles = [
    house_1_load,
    house_2_load,
    house_3_load,
    house_4_load,
    house_5_load,
    pv_sunny_large,
    pv_sunny_med,
    pv_cloudy,
    pv_none]

profile_lengths = set([len(profile) for profile in profiles])


if len(profile_lengths) > 1 :
    raise ValueError("Profiles are not the same length")
else :
    print("All Load and Generation Profiles are the same length")

print("Number of Simulation steps=", num_steps)
print("Number of Hours Simulated=", simulation_hours) 

# Define battery size parameters for small, med and large classs batteries, this can later be matched to real battery models

SMALL_BATTERY = {
    "capacity_kwh": 5.0,
    "initial_soc_kwh": 0.5,
    "max_charge_kw": 2.0,
    "max_discharge_kw": 2.0,
    "efficiency": 0.95,
    "min_soc_kwh": 0.5
}

MEDIUM_BATTERY = {
    "capacity_kwh": 10.0,
    "initial_soc_kwh": 1.0,
    "max_charge_kw": 4.0,
    "max_discharge_kw": 4.0,
    "efficiency": 0.95,
    "min_soc_kwh": 0.5
}

LARGE_BATTERY = {
    "capacity_kwh": 15.0,
    "initial_soc_kwh": 1.5,
    "max_charge_kw": 5.0,
    "max_discharge_kw": 5.0,
    "efficiency": 0.95,
    "min_soc_kwh": 0.5
}



All Load and Generation Profiles are the same length
Number of Simulation steps= 24
Number of Hours Simulated= 24.0


In [29]:
# create empty list to store results summary results and individual house tracking
simulation_results_summary = []
house_results_log = {
    "House 1": [],
    "House 2": [],
    "House 3": [],
    "House 4": [],
    "House 5": [],
}

#create empty counter
count = 0

# initialise the VPP controller to test all modes
controller = VPPController(
    default_mode="peak_shaving",
    peak_hours=[16,17,18,19,20],
    event_hours=[],
    cheap_energy_hours=[3,4,5,6]
)
scenario_name="peakshav_and_arbitrage"


# Initialise Batteries
house1_battery = Battery(**LARGE_BATTERY)
house2_battery = Battery(**SMALL_BATTERY)
house4_battery = Battery(**MEDIUM_BATTERY)

# Initialise Houses
houses = [
    House("House 1", house_1_load, pv_sunny_large, house1_battery),
    House("House 2", house_2_load, pv_sunny_med, house2_battery),
    House("House 3", house_3_load, pv_sunny_large),
    House("House 4", house_4_load, pv_sunny_med, house4_battery),
    House("House 5", house_5_load, pv_none),
]


# Initialise hospital and wind turbine

hospital = LocalLoad("Hospital", 8.0)
wind = WindTurbine("DFIG Wind Placeholder", 15.0)

for t in range(num_steps):
    count += 1

    signals = controller.get_signals(t)
    
    house_results = [house.step(t, timestep_hours, signals) for house in houses]
    
    hospital_result = hospital.step(t)
    wind_result = wind.step(t)

    
    # determine summary results for each step i.e totals for each group of objects 
    total_house_demand_kw = sum(h["load_kw"] for h in house_results)
    total_house_generation_kw = sum(h["generation_kw"] for h in house_results)
    total_house_net_power_kw = sum(h["net_power_kw"] for h in house_results)
    total_house_net_power_before_battery_kw = sum(h["net_power_kw_before_battery"] for h in house_results)
    total_battery_power_kw = sum(h["battery_power_kw"] for h in house_results)
    total_battery_soc_kwh = sum(h["battery_soc_kwh"] for h in house_results)
    total_local_load = hospital_result["load_kw"]
    total_wind_gen = wind_result["generation_kw"]
    VPP_mode = signals["mode"]
    is_peak_period = signals["is_peak_period"]
    is_event_period = signals["is_event_period"]
    is_cheap_energy_period = signals["is_cheap_energy_period"]
    total_battery_discharge_kw = sum(h["battery_discharge_kw"] for h in house_results)
    total_battery_charge_kw = sum(h["battery_charge_kw"] for h in house_results)

    

    
    net_grid_power_kw = total_house_net_power_kw + total_local_load - total_wind_gen

    
    grid_import_kw = max(net_grid_power_kw, 0.0)
    grid_export_kw = -min(net_grid_power_kw, 0.0)

   
    # create set of the summary results 
    timestep_result = {
                        "Timestep": t,
                        "House_demand_kw": total_house_demand_kw,
                        "House_generation_kw": total_house_generation_kw,
                        "House_net_power_before_battery_kw": total_house_net_power_before_battery_kw,
                        "Battery_power_kw": total_battery_power_kw,
                        "Battery_soc_kwh": total_battery_soc_kwh,
                        "House_net_power_kw": total_house_net_power_kw,
                        "Hospital_load_kw": total_local_load,
                        "Wind_generation_kw": total_wind_gen,
                        "Net_grid_power_kw": net_grid_power_kw,
                        "Grid_import_kw": grid_import_kw,
                        "Grid_export_kw": grid_export_kw,
                        "Battery_discharge_kwh": total_battery_discharge_kw * timestep_hours,
                        "Battery_charge_kwh": total_battery_charge_kw * timestep_hours,
                        "VPP_mode": VPP_mode,
                        "Is_peak_period": is_peak_period,
                        "Is_event_period": is_event_period,
                        "Is_cheap_energy_period": is_cheap_energy_period,
                        "Battery_discharge_kw": total_battery_discharge_kw,
                        "Battery_charge_kw": total_battery_charge_kw,
                      }
    # append summary results for this timestep to the simulation summary list
    simulation_results_summary.append(timestep_result)

    # gather and store result for each house indivudually per timestep
    for result in house_results:
        house_name = result["name"]
        house_results_log[house_name].append(result)
        

# create timestape filename
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
# filename = f"SimulationResults_{timestamp}.xlsx"  

filename = f"SimulationResults_{scenario_name}_{timestamp}.xlsx"
filepath = f"results/{filename}"


# convert the simulation summary results list to dataframe
summary_df = pd.DataFrame(simulation_results_summary).round(2)

with pd.ExcelWriter(filepath, engine="openpyxl") as writer:
    summary_df.to_excel(writer, sheet_name="Summary", index=False)

    for house_name, house_log in house_results_log.items():
        house_df = pd.DataFrame(house_log).round(2)
        sheet_name = house_name.replace(" ", "_")
        house_result_export = house_df.drop(columns=["name"])
        house_result_export.to_excel(writer, sheet_name=sheet_name, index=False)



# verify simulation ran correct amount of expected steps and results were exported
print(count,"simulation steps completed")  
print(f"Results exported to results/{filename}")


24 simulation steps completed
Results exported to results/SimulationResults_peakshav_and_arbitrage_2026-05-28_01-25.xlsx


In [55]:
# # battery class testing
# battery_test = Battery(
#     capacity_kwh=10,
#     initial_soc_kwh=5.0,
#     max_charge_kw=2.0,
#     max_discharge_kw=2.0,
#     efficiency=0.95,
#     min_soc_kwh=1.0
# )



# timestep_hours = 1.0

# print(round(battery_test.get_soc_percent(),2),"%")
# charge_test = battery_test.charge(2.0, timestep_hours)
# print(round(charge_test,2))
# print(round(battery_test.get_soc_percent(),2),"%")


# discharge_test=battery_test.discharge(4.0, timestep_hours)
# print(round(discharge_test,2))
# print(round(battery_test.get_soc_percent(),2),"%")




50.0 %
2.0
69.0 %
2.0
47.95 %
